In [1]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import re
import calendar

Find images within a certain data range and ensure we are getting clear image. The metrics we are checking are cloud cover, no data percentage, and sun elevation.

In [ ]:
# Set up parameters
bucket_name = "sentinel-cogs"
base_prefix = "sentinel-s2-l2a-cogs"
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Generate all possible tiles
utm_zones = range(1, 61)  # 1-60
latitude_bands = 'CDEFGHJKLMNPQRSTUVWX'  # Sentinel-2 latitude bands
grid_squares = [f"{x}{y}" for x in 'ABCDEFGHIJKLMNOPQRSTUVWXZ' for y in 'ABCDEFGHIJKLMNOPQRSTUVWXZ']

def check_path_exists(path):
    try:
        response = s3.list_objects_v2(
            Bucket=bucket_name,
            Prefix=path,
            MaxKeys=1
        )
        return 'Contents' in response
    except Exception as e:
        print(f"Error checking path {path}: {e}")
        return False

# Process each day of 2024
current_date = datetime(2024, 1, 1)
end_of_2024 = datetime(2024, 1, 30)

while current_date <= end_of_2024:
    print(f"\nProcessing date: {current_date.strftime('%Y-%m-%d')}")
    
    year = current_date.strftime('%Y')
    month = current_date.strftime('%-m')  # Remove leading zero
    day = current_date.strftime('%d')
    date_str = current_date.strftime('%Y%m%d')
    
    found_paths = []
    
    # Check each possible tile combination
    for utm in utm_zones:
        for lat_band in latitude_bands:
            for grid in grid_squares:
                # Construct the base path for this tile
                tile_id = f"{utm}{lat_band}{grid}"
                
                # Check both Sentinel-2A and 2B
                for satellite in ['S2A', 'S2B']:
                    # Construct the scene ID path
                    scene_id = f"{satellite}_{tile_id}_{date_str}"
                    path = f"{base_prefix}/{utm}/{lat_band}/{grid}/{year}/{month}/{scene_id}"
                    
                    if check_path_exists(path):
                        found_paths.append(path)
                        print(f"Found valid path: {path}")
    
    print(f"Total paths found for {current_date.strftime('%Y-%m-%d')}: {len(found_paths)}")
    
    # Move to next day
    current_date += timedelta(days=1)

In [ ]:
# Initialize variables to track valid folders
valid_folders = []

# Define only the most critical thresholds
THRESHOLDS = {
    'cloud_cover': 10.0,     # Less than 10%
    'nodata_percentage': 5.0, # Less than 5%
    'sun_elevation': 20.0     # Greater than 20 degrees
}

def check_image_quality(metadata):
    """
    Check if the image meets critical quality thresholds.
    Returns tuple of (bool, dict of property values)
    """
    properties = metadata.get('properties', {})
    
    # Extract only critical properties
    values = {
        'cloud_cover': properties.get('eo:cloud_cover'),
        'nodata_percentage': properties.get('s2:nodata_pixel_percentage'),
        'sun_elevation': properties.get('view:sun_elevation')
    }
    
    # Check if any required values are missing
    if any(v is None for v in values.values()):
        return False, values
    
    # Check critical thresholds
    meets_thresholds = (
        values['cloud_cover'] <= THRESHOLDS['cloud_cover'] and
        values['nodata_percentage'] <= THRESHOLDS['nodata_percentage'] and
        values['sun_elevation'] >= THRESHOLDS['sun_elevation']
    )
    
    return meets_thresholds, values

# Main processing loop
for folder in folders:
    print("\nProcessing folder:", folder)
    resp = s3.list_objects_v2(Bucket=bucket_name, Prefix=folder)
    json_files = [obj['Key'] for obj in resp.get('Contents', []) if obj['Key'].endswith('.json')]
    
    if not json_files:
        print("No JSON file found in folder.")
        continue
    
    json_key = json_files[0]
    print("Found JSON file:", json_key)
    
    try:
        obj_response = s3.get_object(Bucket=bucket_name, Key=json_key)
        json_content = obj_response['Body'].read().decode('utf-8')
        metadata = json.loads(json_content)
    except Exception as e:
        print("Error reading JSON from folder:", e)
        continue

    # Check if image meets quality thresholds
    meets_thresholds, values = check_image_quality(metadata)
    
    # Print critical metrics
    print("\nImage Quality Metrics:")
    print(f"Cloud Cover: {values['cloud_cover']}%")
    print(f"No Data Pixels: {values['nodata_percentage']}%")
    print(f"Sun Elevation: {values['sun_elevation']}°")
    print(f"Meets Thresholds: {'Yes' if meets_thresholds else 'No'}")

    if meets_thresholds:
        valid_folders.append(folder)
        print("→ Valid image found!")

# Print final results
print("\n=== Final Results ===")
print(f"Number of valid folders found: {len(valid_folders)}")
if valid_folders:
    print("\nValid folders:")
    for folder in valid_folders:
        print(f"- {folder}")
else:
    print("\nNo folders met quality thresholds.")